# HW9 - PySpark ML Classifiers

In [ ]:
from pathlib import Path
import shutil
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

In [ ]:
# Spark session

def find_repo_root(start: Path) -> Path:
    """Walk upward until we find the repository root that contains README.md."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing README.md')


repo_root = find_repo_root(Path.cwd())
models_dir = repo_root / 'SavedModels' / 'sparkml'
models_dir.mkdir(parents=True, exist_ok=True)

train_ratings_path = repo_root / 'Previous' / 'HW9' / 'traindata' / 'trainItem.data'
ground_truth_path = repo_root / 'Previous' / 'HW9' / 'testdata' / 'test2_new.txt'
submission_output_path = repo_root / 'submission_sparkml_best.csv'

# Keep Spark workers on the same Python interpreter as this notebook.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Prefer JDK 17 when available because newer Java versions can break Spark locally.
candidate_jdk_paths = [
    Path(r'C:\Program Files\Java\jdk-17'),
    Path(r'C:\Program Files\Eclipse Adoptium\jdk-17'),
    Path(r'C:\Program Files\Microsoft\jdk-17'),
]
for candidate_jdk in candidate_jdk_paths:
    if candidate_jdk.exists():
        os.environ['JAVA_HOME'] = str(candidate_jdk)
        os.environ['PATH'] = str(candidate_jdk / 'bin') + os.pathsep + os.environ['PATH']
        break

spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("HW9_SparkML_MusicRecommender")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


print('Repo root:', repo_root)
print('Models will be saved to:', models_dir)
print('Submission path:', submission_output_path)


Repo root: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender
Models will be saved to: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\SavedModels\sparkml
Submission path: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\submission_sparkml_best.csv


In [ ]:
# Helper readers

def parse_candidate_file(path: Path):
    rows = []
    current_user = None
    remaining = 0

    with path.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line:
                continue

            if '|' in line:
                user_id_str, count_str = line.split('|')
                current_user = int(user_id_str)
                remaining = int(count_str)
            else:
                if current_user is None or remaining <= 0:
                    raise ValueError(f'Unexpected candidate line without user header: {line}')
                rows.append((current_user, int(line)))
                remaining -= 1

    if remaining != 0:
        raise ValueError('Candidate file ended before reading the expected number of tracks.')

    return rows


train_schema = StructType([
    StructField('UserID', IntegerType(), False),
    StructField('ItemID', IntegerType(), False),
    StructField('Rating', DoubleType(), False),
])

ground_truth_schema = StructType([
    StructField('UserID', IntegerType(), False),
    StructField('TrackID', IntegerType(), False),
    StructField('label', DoubleType(), False),
])

track_schema = StructType(
    [
        StructField('TrackID', IntegerType(), False),
        StructField('AlbumID', IntegerType(), True),
        StructField('ArtistID', IntegerType(), True),
    ]
    + [StructField(f'Genre{i}', IntegerType(), True) for i in range(1, 22)]
)

genre_cols = [f'Genre{i}' for i in range(1, 22)]


In [ ]:
# Load the labeled data, final candidates, metadata, and fetched ratings

ground_truth_df = (
    spark.read.csv(
        str(ground_truth_path),
        sep='|',
        schema=ground_truth_schema,
        header=False,
    )
    .cache()
)

# Use the prepared CSV files
train_ratings_df = (
    spark.read.csv(
        str(repo_root / 'Assets' / 'CSV' / 'train_data.csv'),
        header=True,
        inferSchema=True,
    )
    .select(
        F.col('UserID').cast('int'),
        F.col('ItemID').cast('int'),
        F.col('Rating').cast('double'),
    )
    .cache()
)

final_candidates_df = (
    spark.read.csv(
        str(repo_root / 'Assets' / 'CSV' / 'test_data.csv'),
        header=True,
        inferSchema=True,
    )
    .select(
        F.col('UserID').cast('int'),
        F.col('TrackID').cast('int'),
    )
    .cache()
)

track_meta_df = (
    spark.read.csv(
        str(repo_root / 'Assets' / 'CSV' / 'track_data.csv'),
        header=True,
        inferSchema=True,
    )
    .select(
        F.col('TrackID').cast('int'),
        F.col('AlbumID').cast('int'),
        F.col('ArtistID').cast('int'),
        *[F.col(f'Genre{i}').cast('int') for i in range(1, 22)],
    )
    .cache()
)

all_target_users_df = (
    ground_truth_df.select('UserID')
    .union(final_candidates_df.select('UserID'))
    .distinct()
    .cache()
)

fetched_ratings_df = (
    train_ratings_df.join(all_target_users_df, on='UserID', how='inner')
    .groupBy('UserID', 'ItemID')
    .agg(F.avg('Rating').alias('Rating'))
    .cache()
)

## Feature Engineering Strategy
So we build a feature matrix for each `(UserID, TrackID)` pair:
- direct user rating of the candidate track
- direct user rating of the candidate album, artist, and genres
- mean and max ratings on sibling tracks from the same album or same artist
- user profile statistics
- global track, album, and artist popularity statistics
- number of genres attached to the candidate track

In [ ]:
# Build statistics tables from the training ratings

track_only_history_df = (
    fetched_ratings_df.alias('r')
    .join(track_meta_df.alias('t'), F.col('r.ItemID') == F.col('t.TrackID'), 'inner')
    .select(
        F.col('r.UserID').alias('UserID'),
        F.col('t.TrackID').alias('TrackID'),
        F.col('r.Rating').alias('Rating'),
        F.col('t.AlbumID').alias('AlbumID'),
        F.col('t.ArtistID').alias('ArtistID'),
        *[F.col(f't.{genre_col}').alias(genre_col) for genre_col in genre_cols],
    )
    .cache()
)

user_stats_df = (
    fetched_ratings_df.groupBy('UserID')
    .agg(
        F.avg('Rating').alias('user_rating_mean'),
        F.stddev_pop('Rating').alias('user_rating_std'),
        F.count('*').alias('user_rating_count'),
    )
    .cache()
)

track_stats_df = (
    track_only_history_df.groupBy('TrackID')
    .agg(
        F.avg('Rating').alias('track_hist_mean'),
        F.count('*').alias('track_hist_count'),
    )
    .cache()
)

album_stats_df = (
    track_only_history_df.groupBy('AlbumID')
    .agg(
        F.avg('Rating').alias('album_hist_mean'),
        F.count('*').alias('album_hist_count'),
        F.countDistinct('TrackID').alias('album_track_catalog_size'),
    )
    .cache()
)

artist_stats_df = (
    track_only_history_df.groupBy('ArtistID')
    .agg(
        F.avg('Rating').alias('artist_hist_mean'),
        F.count('*').alias('artist_hist_count'),
        F.countDistinct('TrackID').alias('artist_track_catalog_size'),
    )
    .cache()
)

track_direct_df = fetched_ratings_df.select(
    'UserID',
    F.col('ItemID').alias('TrackID'),
    F.col('Rating').alias('user_track_direct_rating'),
)

album_direct_df = fetched_ratings_df.select(
    'UserID',
    F.col('ItemID').alias('AlbumID'),
    F.col('Rating').alias('user_album_direct_rating'),
)

artist_direct_df = fetched_ratings_df.select(
    'UserID',
    F.col('ItemID').alias('ArtistID'),
    F.col('Rating').alias('user_artist_direct_rating'),
)

In [ ]:
# Feature-builder function for any candidate table with columns UserID and TrackID

def build_candidate_feature_table(candidates_df):
    base_df = (
        candidates_df.join(track_meta_df, on='TrackID', how='left')
        .withColumn(
            'genre_count',
            sum(
                (F.when(F.col(genre_col).isNotNull(), F.lit(1)).otherwise(F.lit(0)) for genre_col in genre_cols),
                F.lit(0),
            ),
        )
        .cache()
    )

    # Direct hierarchy ratings in the shared ItemID space.
    direct_df = (
        base_df
        .join(track_direct_df, on=['UserID', 'TrackID'], how='left')
        .join(album_direct_df, on=['UserID', 'AlbumID'], how='left')
        .join(artist_direct_df, on=['UserID', 'ArtistID'], how='left')
    )

    # Direct genre ratings
    candidate_genres_df = (
        base_df.select('UserID', 'TrackID', F.array(*[F.col(genre_col) for genre_col in genre_cols]).alias('GenreArray'))
        .select('UserID', 'TrackID', F.explode_outer('GenreArray').alias('GenreID'))
        .where(F.col('GenreID').isNotNull())
    )

    user_genre_direct_df = (
        candidate_genres_df.alias('cg')
        .join(
            fetched_ratings_df.alias('r'),
            (F.col('cg.UserID') == F.col('r.UserID')) & (F.col('cg.GenreID') == F.col('r.ItemID')),
            'left',
        )
        .groupBy(F.col('cg.UserID').alias('UserID'), F.col('cg.TrackID').alias('TrackID'))
        .agg(
            F.avg('r.Rating').alias('user_genre_direct_mean'),
            F.max('r.Rating').alias('user_genre_direct_max'),
            F.count('r.Rating').alias('user_genre_direct_count'),
        )
    )

    # Ratings on other tracks from the same album by the same user.
    album_sibling_df = (
        base_df.select('UserID', 'TrackID', 'AlbumID').alias('c')
        .join(
            track_only_history_df.select(
                F.col('UserID').alias('hist_UserID'),
                F.col('TrackID').alias('hist_TrackID'),
                F.col('AlbumID').alias('hist_AlbumID'),
                F.col('Rating').alias('hist_Rating'),
            ).alias('h'),
            (F.col('c.UserID') == F.col('h.hist_UserID'))
            & (F.col('c.AlbumID') == F.col('h.hist_AlbumID'))
            & (F.col('c.TrackID') != F.col('h.hist_TrackID')),
            'left',
        )
        .groupBy(F.col('c.UserID').alias('UserID'), F.col('c.TrackID').alias('TrackID'))
        .agg(
            F.avg('h.hist_Rating').alias('user_album_sibling_mean'),
            F.max('h.hist_Rating').alias('user_album_sibling_max'),
            F.count('h.hist_Rating').alias('user_album_sibling_count'),
        )
    )

    # Ratings on other tracks from the same artist by the same user.
    artist_sibling_df = (
        base_df.select('UserID', 'TrackID', 'ArtistID').alias('c')
        .join(
            track_only_history_df.select(
                F.col('UserID').alias('hist_UserID'),
                F.col('TrackID').alias('hist_TrackID'),
                F.col('ArtistID').alias('hist_ArtistID'),
                F.col('Rating').alias('hist_Rating'),
            ).alias('h'),
            (F.col('c.UserID') == F.col('h.hist_UserID'))
            & (F.col('c.ArtistID') == F.col('h.hist_ArtistID'))
            & (F.col('c.TrackID') != F.col('h.hist_TrackID')),
            'left',
        )
        .groupBy(F.col('c.UserID').alias('UserID'), F.col('c.TrackID').alias('TrackID'))
        .agg(
            F.avg('h.hist_Rating').alias('user_artist_sibling_mean'),
            F.max('h.hist_Rating').alias('user_artist_sibling_max'),
            F.count('h.hist_Rating').alias('user_artist_sibling_count'),
        )
    )

    feature_df = (
        direct_df
        .join(user_stats_df, on='UserID', how='left')
        .join(track_stats_df, on='TrackID', how='left')
        .join(album_stats_df, on='AlbumID', how='left')
        .join(artist_stats_df, on='ArtistID', how='left')
        .join(user_genre_direct_df, on=['UserID', 'TrackID'], how='left')
        .join(album_sibling_df, on=['UserID', 'TrackID'], how='left')
        .join(artist_sibling_df, on=['UserID', 'TrackID'], how='left')
        .withColumn('has_track_direct', F.when(F.col('user_track_direct_rating').isNotNull(), 1.0).otherwise(0.0))
        .withColumn('has_album_direct', F.when(F.col('user_album_direct_rating').isNotNull(), 1.0).otherwise(0.0))
        .withColumn('has_artist_direct', F.when(F.col('user_artist_direct_rating').isNotNull(), 1.0).otherwise(0.0))
        .withColumn('has_genre_direct', F.when(F.col('user_genre_direct_count') > 0, 1.0).otherwise(0.0))
        .withColumn('has_album_sibling', F.when(F.col('user_album_sibling_count') > 0, 1.0).otherwise(0.0))
        .withColumn('has_artist_sibling', F.when(F.col('user_artist_sibling_count') > 0, 1.0).otherwise(0.0))
    )

    numeric_feature_cols = [
        'genre_count',
        'user_rating_mean',
        'user_rating_std',
        'user_rating_count',
        'track_hist_mean',
        'track_hist_count',
        'album_hist_mean',
        'album_hist_count',
        'album_track_catalog_size',
        'artist_hist_mean',
        'artist_hist_count',
        'artist_track_catalog_size',
        'user_track_direct_rating',
        'user_album_direct_rating',
        'user_artist_direct_rating',
        'user_genre_direct_mean',
        'user_genre_direct_max',
        'user_genre_direct_count',
        'user_album_sibling_mean',
        'user_album_sibling_max',
        'user_album_sibling_count',
        'user_artist_sibling_mean',
        'user_artist_sibling_max',
        'user_artist_sibling_count',
        'has_track_direct',
        'has_album_direct',
        'has_artist_direct',
        'has_genre_direct',
        'has_album_sibling',
        'has_artist_sibling',
    ]

    # Fill missing values with 0
    feature_df = feature_df.fillna(0, subset=numeric_feature_cols)

    for feature_col in numeric_feature_cols:
        feature_df = feature_df.withColumn(feature_col, F.col(feature_col).cast('double'))

    selected_cols = ['UserID', 'TrackID'] + numeric_feature_cols
    return feature_df.select(*selected_cols), numeric_feature_cols


In [ ]:
# Build the labeled feature matrix and the final test feature matrix

labeled_features_df, feature_cols = build_candidate_feature_table(ground_truth_df.select('UserID', 'TrackID'))
labeled_features_df = (
    labeled_features_df.join(ground_truth_df, on=['UserID', 'TrackID'], how='inner')
    .cache()
)

final_features_df, _ = build_candidate_feature_table(final_candidates_df)
final_features_df = final_features_df.cache()

print('Number of model features:', len(feature_cols))
print('Labeled feature rows:', labeled_features_df.count())
print('Final feature rows:', final_features_df.count())

labeled_features_df.show(5, truncate=False)
print('Feature tables are ready for model training and submission generation.')


Number of model features: 30
Labeled feature rows: 6000
Final feature rows: 120000
+------+-------+-----------+------------------+------------------+-----------------+------------------+----------------+------------------+----------------+------------------------+-----------------+-----------------+-------------------------+------------------------+------------------------+-------------------------+----------------------+---------------------+-----------------------+-----------------------+----------------------+------------------------+------------------------+-----------------------+-------------------------+----------------+----------------+-----------------+----------------+-----------------+------------------+-----+
|UserID|TrackID|genre_count|user_rating_mean  |user_rating_std   |user_rating_count|track_hist_mean   |track_hist_count|album_hist_mean   |album_hist_count|album_track_catalog_size|artist_hist_mean |artist_hist_count|artist_track_catalog_size|user_track_direct_rating|u

## Model Training

We will train:
- LogisticRegression
- DecisionTreeClassifier
- RandomForestClassifier
- GBTClassifier

Validation is done by splitting at the user level, not by random row, so the six rows for the same user do not leak between train and validation.

Workflow used below:
- Train each model on a user-level training split from `test2_new.txt`
- Evaluate each model on a held-out user-level validation split from `test2_new.txt`
- Generate one Kaggle submission file per split-trained classifier

In [ ]:
# Train, evaluate, and generate one submission file per classifier

seed = 42

distinct_user_df = labeled_features_df.select('UserID').distinct()
train_user_df, valid_user_df = distinct_user_df.randomSplit([0.8, 0.2], seed=seed)

train_df = labeled_features_df.join(train_user_df, on='UserID', how='inner').cache()
valid_df = labeled_features_df.join(valid_user_df, on='UserID', how='inner').cache()

print('Training rows:', train_df.count())
print('Validation rows:', valid_df.count())

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

estimators = {
    'logistic_regression': LogisticRegression(
        featuresCol='features',
        labelCol='label',
        maxIter=50,
        regParam=0.05,
        elasticNetParam=0.0,
    ),
    'decision_tree': DecisionTreeClassifier(
        featuresCol='features',
        labelCol='label',
        maxDepth=6,
        minInstancesPerNode=20,
        seed=seed,
    ),
    'random_forest': RandomForestClassifier(
        featuresCol='features',
        labelCol='label',
        numTrees=120,
        maxDepth=8,
        seed=seed,
    ),
    'gbt': GBTClassifier(
        featuresCol='features',
        labelCol='label',
        maxIter=30,
        maxDepth=5,
        seed=seed,
    ),
}

evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC',
)


def evaluate_predictions(predictions_df):
    scored_df = predictions_df.withColumn('score', vector_to_array('probability')[1])
    ranking_window = Window.partitionBy('UserID').orderBy(F.desc('score'), F.asc('TrackID'))
    forced_df = (
        scored_df
        .withColumn('rank_within_user', F.row_number().over(ranking_window))
        .withColumn('forced_top3_prediction', F.when(F.col('rank_within_user') <= 3, 1.0).otherwise(0.0))
    )

    forced_top3_accuracy = forced_df.select(
        F.avg((F.col('forced_top3_prediction') == F.col('label')).cast('double')).alias('forced_top3_accuracy')
    ).first()['forced_top3_accuracy']

    exact_user_match = (
        forced_df.groupBy('UserID')
        .agg(F.min((F.col('forced_top3_prediction') == F.col('label')).cast('int')).alias('all_six_correct'))
        .agg(F.avg('all_six_correct').alias('exact_user_match'))
        .first()['exact_user_match']
    )

    return forced_df, forced_top3_accuracy, exact_user_match


def make_submission_from_predictions(predictions_df, output_path):
    scored_df = predictions_df.withColumn('score', vector_to_array('probability')[1])
    rank_window = Window.partitionBy('UserID').orderBy(F.desc('score'), F.asc('TrackID'))
    submission_df = (
        scored_df
        .withColumn('rank_within_user', F.row_number().over(rank_window))
        .withColumn('Predictor', F.when(F.col('rank_within_user') <= 3, 1).otherwise(0))
        .select(
            F.concat_ws('_', F.col('UserID').cast('string'), F.col('TrackID').cast('string')).alias('TrackID'),
            'Predictor',
        )
    )
    submission_pdf = submission_df.toPandas()
    submission_pdf.to_csv(output_path, index=False)
    return submission_df


validation_metrics = []
validation_predictions = {}
split_trained_models = {}
submission_paths = {}

for model_name, estimator in estimators.items():
    pipeline = Pipeline(stages=[assembler, estimator])
    fitted_pipeline = pipeline.fit(train_df)
    split_trained_models[model_name] = fitted_pipeline
    predictions_df = fitted_pipeline.transform(valid_df).cache()

    auc_value = evaluator.evaluate(predictions_df)
    forced_df, top3_acc, exact_user = evaluate_predictions(predictions_df)

    validation_metrics.append({
        'model_name': model_name,
        'auc': auc_value,
        'forced_top3_accuracy': top3_acc,
        'exact_user_match': exact_user,
    })
    validation_predictions[model_name] = forced_df

    final_test_predictions_df = fitted_pipeline.transform(final_features_df)
    submission_path = repo_root / f'submission_sparkml_validation_{model_name}.csv'
    make_submission_from_predictions(final_test_predictions_df, submission_path)
    submission_paths[model_name] = submission_path
    print(f'Saved validation-phase submission for {model_name}: {submission_path}')

metrics_df = spark.createDataFrame(validation_metrics).orderBy(
    F.desc('forced_top3_accuracy'),
    F.desc('auc'),
)
metrics_df.show(truncate=False)

metrics_output_path = models_dir / 'validation_metrics.csv'
metrics_pdf = metrics_df.toPandas()
metrics_pdf.to_csv(metrics_output_path, index=False)
print('Saved validation metrics to:', metrics_output_path)

best_model_name = metrics_df.first()['model_name']
print('Best validation model:', best_model_name)
print('Submission files created for all four split-trained models:')
for model_name, submission_path in submission_paths.items():
    print(f'  {model_name}: {submission_path}')


Training rows: 5028
Validation rows: 972
Saved validation-phase submission for logistic_regression: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\submission_sparkml_validation_logistic_regression.csv
Saved validation-phase submission for decision_tree: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\submission_sparkml_validation_decision_tree.csv
Saved validation-phase submission for random_forest: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\submission_sparkml_validation_random_forest.csv
Saved validation-phase submission for gbt: C:\Users\ahmad\OneDrive\Desktop\SchoolProjects\BigData_Music_Recommender\submission_sparkml_validation_gbt.csv
+------------------+------------------+--------------------+-------------------+
|auc               |exact_user_match  |forced_top3_accuracy|model_name         |
+------------------+------------------+--------------------+-------------------+
|0.9496858541211536|0.746913